In [2]:
import os
import pickle
import pandas as pd

from sentence_transformers import SentenceTransformer

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [1]:
pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [3]:
df = pd.read_csv("fraudshield_dataset.csv")

print(df.head())
print(df.shape)

                                                text  label           intent
0  Your Aadhaar number has been linked to a money...      1   Fear Induction
1  Do not disconnect this video call or legal act...      1        Isolation
2  This is an official investigation by the Cyber...      1  False Authority
3  Your bank accounts are under surveillance beca...      1   Fear Induction
4  You are prohibited from speaking to your famil...      1        Isolation
(510, 3)


In [6]:

import pandas as pd
old = pd.read_csv('fraudshield_dataset.csv')
new = pd.read_csv('new_english_samples.csv')
merged = pd.concat([old, new], ignore_index=True)
merged.to_csv('fraudshield_dataset.csv', index=False)
print(f'Total samples: {len(merged)}')
print(merged['label'].value_counts())


Total samples: 413
label
1    277
0    136
Name: count, dtype: int64


In [20]:
import pandas as pd
old = pd.read_csv('fraudshield_dataset.csv')
new = pd.read_csv('legit_call_transcripts.csv')
merged = pd.concat([old, new], ignore_index=True)
merged.to_csv('fraudshield_dataset.csv', index=False)
print(f'Total samples: {len(merged)}')
print(merged['label'].value_counts())

Total samples: 510
label
1    277
0    233
Name: count, dtype: int64


In [4]:
X = df["text"]
y = df["label"]

print("Number of samples:", len(X))
print(y.value_counts())

Number of samples: 510
label
1    277
0    233
Name: count, dtype: int64


In [5]:
embedding_model_name = "paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = SentenceTransformer(embedding_model_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [6]:
X_embedded = embedding_model.encode(
    X.tolist(),
    show_progress_bar=True
)

print(X_embedded.shape)

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

(510, 384)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_embedded,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(408, 384)
(102, 384)


In [8]:
print(df.columns)

Index(['text', 'label', 'intent'], dtype='object')


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),
    "Linear SVM": LinearSVC(
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42
    )
}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    print(f"{name}: {acc:.4f}")

Logistic Regression: 0.9020
Linear SVM: 0.9118
Random Forest: 0.8824


In [10]:
from sklearn.svm import LinearSVC

classifier = LinearSVC(
    class_weight="balanced",
    random_state=42
)

classifier.fit(X_train, y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,verbose,0
,random_state,42


In [11]:
y_pred = classifier.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.83      0.90        47
           1       0.87      0.98      0.92        55

    accuracy                           0.91       102
   macro avg       0.92      0.91      0.91       102
weighted avg       0.92      0.91      0.91       102



In [12]:
os.makedirs("model", exist_ok=True)

In [13]:
with open("model/scam_classifier_v3.pkl", "wb") as f:
    pickle.dump(classifier, f)

print("Classifier saved!")

Classifier saved!


In [14]:
with open("model/embedding_model_name.pkl", "wb") as f:
    pickle.dump(embedding_model_name, f)

print("Embedding model name saved!")

Embedding model name saved!


In [15]:
print(os.listdir("model"))

['.ipynb_checkpoints', 'embedding_model_name.pkl', 'scam_classifier_v2.pkl', 'scam_classifier_v3.pkl']
